# WER Evaluation Pipeline — Multi-Dialect Vietnamese TTS

Evaluate TTS quality across **6 checkpoints** and **3 dialects** by:
1. Generating speech with F5-TTS for each test transcript × dialect
2. Transcribing generated audio with PhoWhisper Medium ASR
3. Computing Word Error Rate (WER) against original transcripts

**Total: 50 samples × 3 dialects × 4 checkpoints = 600 generations**

## 1. Setup & Install

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%cd /content/drive/MyDrive/mdv-tts/F5-TTS
!pip install -e . -q
!pip install jiwer -q

/content/drive/MyDrive/mdv-tts/F5-TTS
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.4/101.4 kB 9.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 112.8 MB/s eta 0:00:0000:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 52.4 MB/s eta 0:00:00:00:010:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.0/43.0 MB 61.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.2/59.2 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.5/107.5 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 840.2/840.2 kB 90.9 MB/s eta 0:00:00
   ━━

## 2. Configuration

In [1]:
import os

DRIVE_ROOT = "/content/drive/MyDrive/mdv-tts"
TESTS_CSV = os.path.join(DRIVE_ROOT, "tests", "tests.csv")
VOCAB_FILE = os.path.join(DRIVE_ROOT, "F5-TTS/data/Emilia_ZH_EN_pinyin/vocab.txt")
CKPT_DIR = os.path.join(DRIVE_ROOT, "F5-TTS/ckpts/viet")
OUTPUT_DIR = "/content/wer_results"
DRIVE_RESULTS = os.path.join(DRIVE_ROOT, "tests", "results")

CHECKPOINTS = [20000, 30000, 35000, 40000]
DIALECTS = ["North", "Central", "South"]

# Reference audio & text per dialect (same config as mdv_tts_app.py)
DIALECT_CONFIG = {
    "North": {
        "ref_audio": os.path.join(DRIVE_ROOT, "references/north_ref.wav"),
        "ref_text": (
            "[North] ý tưởng đầu tiên tôi phải nhắc đến là vấn đề về chi phí xây dựng phần mềm biết cốt theo truyền thống."
        ),
    },
    "Central": {
        "ref_audio": os.path.join(DRIVE_ROOT, "references/central_ref.wav"),
        "ref_text": (
            "[Central] các cái ki bốt này được đầu tư từ những năm hai không mười hai không mười ba tức là cách đây cũng khoảng hơn mười năm rồi và quá trình."
        ),
    },
    "South": {
        "ref_audio": os.path.join(DRIVE_ROOT, "references/south_ref.wav"),
        "ref_text": (
            "[South] Chúng tôi tập trung vào cái công tác đào tạo huấn luyện, đặc biệt là huấn luyện về cái quy trình các cái hướng dẫn chẩn đoán điều trị Covid mười chín của Bộ Y tế,"
        ),
    },
}

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(DRIVE_RESULTS, exist_ok=True)

# Verify paths
print(f"Tests CSV exists: {os.path.exists(TESTS_CSV)}")
print(f"Vocab file exists: {os.path.exists(VOCAB_FILE)}")
for name, cfg in DIALECT_CONFIG.items():
    print(f"{name} ref audio exists: {os.path.exists(cfg['ref_audio'])}")
for ckpt in CHECKPOINTS:
    p = os.path.join(CKPT_DIR, f"model_{ckpt}.safetensors")
    print(f"Checkpoint {ckpt} exists: {os.path.exists(p)}")

Tests CSV exists: True
Vocab file exists: True
North ref audio exists: True
Central ref audio exists: True
South ref audio exists: True
Checkpoint 20000 exists: True
Checkpoint 30000 exists: True
Checkpoint 35000 exists: True
Checkpoint 40000 exists: True


## 3. Load PhoWhisper ASR Model

In [2]:
import torch
import librosa
from transformers import WhisperProcessor, WhisperForConditionalGeneration

print("Loading PhoWhisper Medium...")
asr_processor = WhisperProcessor.from_pretrained("vinai/PhoWhisper-medium")
asr_model = WhisperForConditionalGeneration.from_pretrained(
    "vinai/PhoWhisper-medium", torch_dtype=torch.float16
).to("cuda")
print("PhoWhisper loaded!")


Loading PhoWhisper Medium...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading weights:   0%|          | 0/948 [00:00<?, ?it/s]

Exception in thread Thread-auto_conversion:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_http.py", line 761, in hf_raise_for_status
    response.raise_for_status()
  File "/usr/local/lib/python3.12/dist-packages/httpx/_models.py", line 829, in raise_for_status
    raise HTTPStatusError(message, request=request, response=self)
httpx.HTTPStatusError: Client error '403 Forbidden' for url 'https://huggingface.co/api/models/vinai/PhoWhisper-medium/discussions?p=0'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/403

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.

PhoWhisper loaded!


## 4. Utility Functions

In [3]:
import re
import csv
import gc
import time
import tempfile
import numpy as np
import soundfile as sf
from jiwer import wer as compute_wer


def normalize_text(text: str) -> str:
    """Normalize text for WER comparison.
    Removes dialect tags, punctuation, extra whitespace, and lowercases.
    """
    text = re.sub(r"\[(North|Central|South)\]\s*", "", text)
    text = re.sub(r"[^\w\s]", "", text)
    text = text.lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text


def load_test_samples(csv_path: str) -> list[dict]:
    """Load test samples from CSV. Returns list of dicts with 'text' key."""
    samples = []
    with open(csv_path, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            if row["text"].strip():
                samples.append({"text": row["text"].strip(), "original_region": row.get("region", "")})
    print(f"Loaded {len(samples)} test samples")
    return samples


def generate_and_transcribe(tts_model, ref_audio, ref_text, gen_text):
    """Generate audio with TTS, then transcribe with ASR. Returns ASR text."""
    if not gen_text.strip():
        return ""
        
    wav, sr, _ = tts_model.infer(
        ref_file=ref_audio,
        ref_text=ref_text,
        gen_text=gen_text,
    )
    
    # Catch empty audio before crashing soundfile
    if len(wav) == 0:
        print(f"    [WARNING] TTS generated 0-length audio for text: {gen_text}")
        return ""
        
    # Save to temp WAV for ASR
    # Transcribe
    # result = asr(tmp_path)
    # Resample to 16kHz for Whisper
    if sr != 16000:
        wav_16k = librosa.resample(wav, orig_sr=sr, target_sr=16000)
    else:
        wav_16k = wav
    input_features = asr_processor(
        wav_16k, sampling_rate=16000, return_tensors="pt"
    ).input_features.to("cuda", dtype=torch.float16)
    predicted_ids = asr_model.generate(input_features)
    result_text = asr_processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
    return result_text


def load_tts_checkpoint(ckpt_step: int):
    """Load F5TTS with a specific checkpoint. Returns the model."""
    from f5_tts.api import F5TTS
    ckpt_path = os.path.join(CKPT_DIR, f"model_{ckpt_step}.safetensors")
    print(f"  Loading checkpoint: {ckpt_path}")
    tts = F5TTS(
        model="F5TTS_v1_Base",
        ckpt_file=ckpt_path,
        vocab_file=VOCAB_FILE,
        device="cuda",
    )
    return tts


def free_tts_memory(tts):
    """Delete TTS model and free GPU memory."""
    del tts
    gc.collect()
    torch.cuda.empty_cache()
    print("  GPU memory cleared.")


# Quick test
samples = load_test_samples(TESTS_CSV)
print(f"Sample 0: {samples[0]['text'][:80]}...")
print(f"Normalized: {normalize_text(samples[0]['text'])[:80]}...")

Loaded 50 test samples
Sample 0: về dự án hỗ trợ thêm cái mô hình cây rong riềng....
Normalized: về dự án hỗ trợ thêm cái mô hình cây rong riềng...


## 5. Main Evaluation Loop

In [4]:
all_results = []  # list of dicts per sample
summary_results = []  # list of dicts per checkpoint x dialect

total_start = time.time()
total_gens = len(CHECKPOINTS) * len(DIALECTS) * len(samples)
gen_count = 0

for ckpt_step in CHECKPOINTS:
    print(f"\n{'='*60}")
    print(f"CHECKPOINT {ckpt_step}")
    print(f"{'='*60}")

    tts = load_tts_checkpoint(ckpt_step)

    for dialect in DIALECTS:
        print(f"\n  --- Dialect: {dialect} ---")
        cfg = DIALECT_CONFIG[dialect]
        ref_audio = cfg["ref_audio"]
        ref_text = cfg["ref_text"]

        dialect_refs = []
        dialect_hyps = []

        for i, sample in enumerate(samples):
            gen_count += 1
            original_text = sample["text"]

            try:
                asr_text = generate_and_transcribe(
                    tts, ref_audio, ref_text, original_text
                )
            except Exception as e:
                import traceback
                print(f"    [ERROR] Sample {i}: {e}")
                traceback.print_exc()  # <--- Add this line
                asr_text = ""

            norm_ref = normalize_text(original_text)
            norm_hyp = normalize_text(asr_text)

            if norm_ref and norm_hyp:
                sample_wer = compute_wer(norm_ref, norm_hyp)
            else:
                sample_wer = 1.0  # treat empty as 100% error

            all_results.append({
                "checkpoint": ckpt_step,
                "dialect": dialect,
                "sample_idx": i,
                "original_region": sample["original_region"],
                "original_text": original_text,
                "asr_text": asr_text,
                "norm_original": norm_ref,
                "norm_asr": norm_hyp,
                "wer": sample_wer,
            })

            dialect_refs.append(norm_ref)
            dialect_hyps.append(norm_hyp)

            if (i + 1) % 10 == 0 or i == len(samples) - 1:
                elapsed = time.time() - total_start
                rate = gen_count / elapsed
                eta = (total_gens - gen_count) / rate if rate > 0 else 0
                print(
                    f"    [{gen_count}/{total_gens}] "
                    f"Sample {i+1}/{len(samples)} | "
                    f"WER: {sample_wer:.2%} | "
                    f"ETA: {eta/60:.1f} min"
                )

        # Compute dialect-level WER
        valid = [(r, h) for r, h in zip(dialect_refs, dialect_hyps) if r and h]
        if valid:
            refs_valid, hyps_valid = zip(*valid)
            dialect_wer = compute_wer(list(refs_valid), list(hyps_valid))
        else:
            dialect_wer = 1.0

        summary_results.append({
            "checkpoint": ckpt_step,
            "dialect": dialect,
            "num_samples": len(samples),
            "num_valid": len(valid),
            "wer": dialect_wer,
        })
        print(f"  >> {dialect} WER @ ckpt {ckpt_step}: {dialect_wer:.2%}")

    free_tts_memory(tts)

total_elapsed = time.time() - total_start
print(f"\n{'='*60}")
print(f"DONE! Total time: {total_elapsed/60:.1f} min ({total_elapsed/3600:.1f} hrs)")
print(f"Total generations: {gen_count}")


CHECKPOINT 20000
  Loading checkpoint: /content/drive/MyDrive/mdv-tts/F5-TTS/ckpts/viet/model_20000.safetensors
Download Vocos from huggingface charactr/vocos-mel-24khz

vocab :  /content/drive/MyDrive/mdv-tts/F5-TTS/data/Emilia_ZH_EN_pinyin/vocab.txt
token :  custom
model :  /content/drive/MyDrive/mdv-tts/F5-TTS/ckpts/viet/model_20000.safetensors 


  --- Dialect: North ---
Converting audio...
Using custom reference text...

ref_text   [North] ý tưởng đầu tiên tôi phải nhắc đến là vấn đề về chi phí xây dựng phần mềm biết cốt theo truyền thống. 
gen_text 0 về dự án hỗ trợ thêm cái mô hình cây rong riềng.


Generating audio in 1 batches...


100%|██████████| 1/1 [00:00<00:00,  1.19it/s]
Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. 

Converting audio...
Using cached preprocessed reference audio...
Using custom reference text...

ref_text   [North] ý tưởng đầu tiên tôi phải nhắc đến là vấn đề về chi phí xây dựng phần mềm biết cốt theo truyền thống. 
gen_text 0 Được tiêm vắc xin thì cũng là điều an tâm an toàn nên em cũng háo hức.


Generating audio in 1 batches...


100%|██████████| 1/1 [00:00<00:00,  2.09it/s]


Converting audio...
Using cached preprocessed reference audio...
Using custom reference text...

ref_text   [North] ý tưởng đầu tiên tôi phải nhắc đến là vấn đề về chi phí xây dựng phần mềm biết cốt theo truyền thống. 
gen_text 0 riêng bản thân mình thì một ngày mình có thể dành tới tám đến mười tiếng đồng hồ. Anh em là trực chốt tới từ sáu giờ sáng.


Generating audio in 1 batches...


...

Converting audio...
Using cached preprocessed reference audio...
Using custom reference text...

ref_text   [South] Chúng tôi tập trung vào cái công tác đào tạo huấn luyện, đặc biệt là huấn luyện về cái quy trình các cái hướng dẫn chẩn đoán điều trị Covid mười chín của Bộ Y tế,. 
gen_text 0 Thứ nhất, hiện nay là cái du lịch cộng đồng đang được coi là loại hình du lịch mang lại nhiều lợi ích phát triển kinh tế cho bà con nhân dân tại địa các địa phương trong tỉnh, nhất là bà con các cái khu các điểm cảnh.


Generating audio in 1 batches...


100%|██████████| 1/1 [00:00<00:00,  1.63it/s]


Converting audio...
Using cached preprocessed reference audio...
Using custom reference text...

ref_text   [South] Chúng tôi tập trung vào cái công tác đào tạo huấn luyện, đặc biệt là huấn luyện về cái quy trình các cái hướng dẫn chẩn đoán điều trị Covid mười chín của Bộ Y tế,. 
gen_text 0 Và có những cái vụ thì cái tổng ở nước ngoài và các tỉnh thành trong cả nước. Và các đối tượng các con bạc thì mua các cái trang mạng về để núp dưới một số cái miền sau đó thì nó tổ chức và nó đánh bạc


Generating audio in 1 batches...


100%|██████████| 1/1 [00:00<00:00,  1.65it/s]


    [600/600] Sample 50/50 | WER: 114.29% | ETA: 0.0 min
  >> South WER @ ckpt 40000: 106.65%
  GPU memory cleared.

DONE! Total time: 15.0 min (0.3 hrs)
Total generations: 600


## 6. Export Results

In [5]:
import pandas as pd

# Detailed results
df_detail = pd.DataFrame(all_results)
detail_path = os.path.join(OUTPUT_DIR, "wer_detailed_results.csv")
df_detail.to_csv(detail_path, index=False)
print(f"Detailed results saved: {detail_path}")

# Summary results
df_summary = pd.DataFrame(summary_results)
summary_path = os.path.join(OUTPUT_DIR, "wer_summary.csv")
df_summary.to_csv(summary_path, index=False)
print(f"Summary results saved: {summary_path}")

# Copy to Drive
import shutil
shutil.copy2(detail_path, DRIVE_RESULTS)
shutil.copy2(summary_path, DRIVE_RESULTS)
print(f"Results copied to Drive: {DRIVE_RESULTS}")

# Display summary table
pivot = df_summary.pivot(index="checkpoint", columns="dialect", values="wer")
pivot["Overall"] = df_detail.groupby("checkpoint")["wer"].mean()
print("\n" + "="*60)
print("WER SUMMARY (lower is better)")
print("="*60)
print(pivot.to_string(float_format="{:.2%}".format))

Detailed results saved: /content/wer_results/wer_detailed_results.csv
Summary results saved: /content/wer_results/wer_summary.csv
Results copied to Drive: /content/drive/MyDrive/mdv-tts/tests/results

WER SUMMARY (lower is better)
dialect     Central   North   South  Overall
checkpoint                                  
20000       118.22% 127.70% 112.80%  118.28%
30000       110.83% 116.74% 108.33%  111.38%
35000       117.97% 121.75% 106.61%  114.22%
40000       104.06% 110.22% 106.65%  106.87%


## 7. Visualization

In [6]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({"font.size": 12, "figure.dpi": 120})

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Line plot: WER across checkpoints per dialect ---
ax1 = axes[0]
colors = {"North": "#4A90D9", "Central": "#E8833A", "South": "#50B86C"}
for dialect in DIALECTS:
    sub = df_summary[df_summary["dialect"] == dialect]
    ax1.plot(
        sub["checkpoint"], sub["wer"] * 100,
        marker="o", linewidth=2, markersize=8,
        label=dialect, color=colors[dialect],
    )
# Overall
overall = df_detail.groupby("checkpoint")["wer"].mean()
ax1.plot(
    overall.index, overall.values * 100,
    marker="s", linewidth=2.5, markersize=8,
    label="Overall", color="#333333", linestyle="--",
)
ax1.set_xlabel("Checkpoint (training steps)")
ax1.set_ylabel("Word Error Rate (%)")
ax1.set_title("WER Across Checkpoints")
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_xticks(CHECKPOINTS)
ax1.set_xticklabels([f"{c//1000}k" for c in CHECKPOINTS])

# --- Heatmap: Checkpoint x Dialect ---
ax2 = axes[1]
pivot_pct = pivot[DIALECTS] * 100
im = ax2.imshow(pivot_pct.values, cmap="RdYlGn_r", aspect="auto")
ax2.set_xticks(range(len(DIALECTS)))
ax2.set_xticklabels(DIALECTS)
ax2.set_yticks(range(len(CHECKPOINTS)))
ax2.set_yticklabels([f"{c//1000}k" for c in CHECKPOINTS])
ax2.set_xlabel("Dialect")
ax2.set_ylabel("Checkpoint")
ax2.set_title("WER Heatmap (%)")
# Annotate cells
for i in range(len(CHECKPOINTS)):
    for j in range(len(DIALECTS)):
        val = pivot_pct.values[i, j]
        ax2.text(j, i, f"{val:.1f}%", ha="center", va="center",
                 fontsize=11, fontweight="bold",
                 color="white" if val > 50 else "black")
plt.colorbar(im, ax=ax2, label="WER (%)")

plt.tight_layout()
fig_path = os.path.join(OUTPUT_DIR, "wer_results.png")
plt.savefig(fig_path, bbox_inches="tight")
shutil.copy2(fig_path, DRIVE_RESULTS)
plt.show()
print(f"Figure saved to: {fig_path}")

Figure saved to: /content/wer_results/wer_results.png
